# 01 — EEGNet (compact CNN baseline)

**Family:** CNN &nbsp;•&nbsp; **Input:** raw windowed signal `[14 x window_samples]`

EEGNet (Lawhern et al., 2018) is the lightweight reference point for the whole
benchmark. It factorizes EEG feature extraction into a temporal convolution
(frequency filters), a depthwise convolution across channels (spatial filters per
frequency band) and a separable convolution (feature-map mixing). It has the
smallest parameter budget of the eight models, which makes it the natural
accuracy-vs-efficiency anchor.

## 1. Setup

In [ ]:
import os, sys

# make the `src` package importable when the notebook runs from notebooks/
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src import benchmark_si as B
from src.evaluation import set_seed, get_device, count_parameters
from src.data_loader import load_subject
from src.models import eegnet

set_seed(42)
DEVICE = get_device()
PROCESSED_DIR = os.path.join(ROOT, "data", "processed")

# ----- experiment configuration -----
MODEL_NAME = "EEGNet"
LABEL = "valence"     # main comparison target. Swap to "arousal" / "dominance" to reproduce.
QUICK = False         # True  -> 3-fold / few-epoch smoke test
                      # False -> the full 23-fold benchmark used for the thesis table
print("device:", DEVICE, "| model:", MODEL_NAME, "| label:", LABEL, "| QUICK:", QUICK)
print("recipe:", B.REGISTRY[MODEL_NAME])


## 2. Architecture

`src/models/eegnet.py` implements EEGNet-8,2 (F1 = 8 temporal filters, depth
multiplier D = 2, F2 = 16 separable filters, temporal kernel = 64 ≈ 0.5 s at
128 Hz). Depthwise kernels carry a max-norm constraint (via `torch.renorm`) as in
the reference implementation.

In [14]:
# Instantiate once on a dummy spec just to print the graph + parameter count.
_dummy_meta = {"input_shape": (14, load_subject(PROCESSED_DIR, "subject_01").window_samples)}
_m = eegnet.build({"n_classes": 2, "meta": _dummy_meta})
print(_m)
print(f"\ntrainable parameters: {count_parameters(_m):,}")
del _m

EEGNet(
  (temporal): Sequential(
    (0): Conv2d(1, 8, kernel_size=(1, 64), stride=(1, 1), padding=(0, 32), bias=False)
    (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (depthwise): Sequential(
    (0): _Constrained2d(8, 16, kernel_size=(14, 1), stride=(1, 1), groups=8, bias=False)
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ELU(alpha=1.0)
    (3): AvgPool2d(kernel_size=(1, 4), stride=(1, 4), padding=0)
    (4): Dropout(p=0.25, inplace=False)
  )
  (separable): Sequential(
    (0): Conv2d(16, 16, kernel_size=(1, 16), stride=(1, 1), padding=(0, 8), groups=16, bias=False)
    (1): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): ELU(alpha=1.0)
    (4): AvgPool2d(kernel_size=(1, 8), stride=(1, 8), padding=0)
    (5): Dropout(p=0.25, inplace=False)
  )
  (classifier): Linear(in_feat

## Validation protocol — subject-independent (LOSO)

The benchmark uses a single protocol so every architecture is scored identically:
**23-fold leave-one-subject-out** (`data/splits/loso_splits.json`). Fold *k* trains a
fresh model on 22 subjects and tests on the held-out 23rd; normalisation statistics
are fit on the training subjects only and a 10% slice of them is used for early
stopping. We report the **mean ± std of the headline metrics across the 23 folds**.

Everything runs through `src.benchmark_si.run_one` — the one entry point shared by
every model notebook — which reads this model's recipe from `benchmark_si.REGISTRY`,
trains through `src.evaluation.run_loso`, and writes `results/{MODEL_NAME}_results.json`.


## 3. Run the subject-independent (LOSO) benchmark

In [ ]:
# One shared call: 23-fold LOSO for this model, then writes results/EEGNet_results.json
res = B.run_one(MODEL_NAME, label=LABEL, quick=QUICK, device=DEVICE)
res

## Notes for the methodology chapter

* Every metric comes from `src/evaluation.compute_metrics` (accuracy, macro
  precision / recall / F1, confusion matrix) — no per-notebook metric code.
* The per-model LOSO recipe (feature view, batch size, epochs, lr) lives in
  `src/benchmark_si.py::REGISTRY` — a single source of truth shared by this
  notebook and the `python -m src.benchmark_si` CLI.
* `params_count` and `train_time_sec` are logged to
  `results/{MODEL_NAME}_results.json` for the accuracy-vs-efficiency analysis.
* Set `QUICK = True` to smoke-test the whole flow (3 folds / few epochs) before
  the full multi-hour run; `LABEL` switches the target between
  valence / arousal / dominance with no other change.
